# Análisis de Resultados — Detección Colaborativa de Meaconing GNSS

**TFM: Arquitectura de Seguridad para Navegación Autónoma de Robots**  
Antonio García Alcón — Universidad Europea de Madrid, 2026

---

Este notebook carga rosbags grabados durante los experimentos y genera las gráficas
para la memoria del TFM:

1. Evolución temporal de $S_k$ para cada escenario
2. TTD vs drift velocity (curva de sensibilidad)
3. Curva ROC (trade-off TTD-FAR)
4. $\Delta(t)$ vs $S(t)$: detector de umbral fijo vs CUSUM

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import sqlite3
import sys
from pathlib import Path
from collections import defaultdict

# rosbag2 reader (requires rosbag2_py available in the ROS 2 environment)
try:
    from rosbag2_py import SequentialReader, StorageOptions, ConverterOptions
    from rclpy.serialization import deserialize_message
    from rosidl_runtime_py.utilities import get_message
    HAS_ROSBAG2 = True
except ImportError:
    print('⚠ rosbag2_py not available. Using SQLite fallback.')
    HAS_ROSBAG2 = False

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print('Notebook ready for TFM analysis.')

## 1. Carga de datos desde rosbag

Cargar un rosbag2 grabado durante un experimento y extraer las series temporales relevantes.

In [ ]:
def load_rosbag(bag_path: str) -> dict:
    """
    Load a rosbag2 file and extract time series for:
    - /system/cusum_value (Float64)
    - /system/delta_value (Float64)
    - /system/meaconing_alert (Bool)
    - /robots/uwb_distance (Float64)
    - /meaconing/active (Bool)
    """
    if not HAS_ROSBAG2:
        print('rosbag2_py not available. Install in your ROS 2 environment.')
        return {}
    
    storage_options = StorageOptions(uri=bag_path, storage_id='sqlite3')
    converter_options = ConverterOptions(
        input_serialization_format='cdr',
        output_serialization_format='cdr'
    )
    
    reader = SequentialReader()
    reader.open(storage_options, converter_options)
    
    # Topic metadata
    type_map = {}
    for topic_meta in reader.get_all_topics_and_types():
        type_map[topic_meta.name] = topic_meta.type
    
    # Store time series
    data = defaultdict(list)
    t0 = None
    
    while reader.has_next():
        (topic, msg_bytes, timestamp_ns) = reader.read_next()
        ts = timestamp_ns / 1e9  # Convert to seconds
        
        if t0 is None:
            t0 = ts
        
        if topic in type_map:
            msg_type = get_message(type_map[topic])
            msg = deserialize_message(msg_bytes, msg_type)
            
            data['time'].append(ts - t0)
            data['topic'].append(topic)
            
            if hasattr(msg, 'data'):
                data['value'].append(msg.data)
            else:
                data['value'].append(np.nan)
    
    return dict(data)

# Example usage:
# data = load_rosbag('experimento_E1')
# print(f"Loaded {len(data['time'])} messages")

## 2. Evolución temporal de S_k

Gráfica de la evolución del estadístico CUSUM para cada escenario de experimento.

In [ ]:
def plot_cusum_evolution(data_by_experiment: dict, tau: float = 2.0, save: str = None):
    """
    Plot CUSUM statistic S_k over time for multiple experiments.
    
    Args:
        data_by_experiment: dict of {label: {'time': [...], 'S_k': [...]}}
        tau: detection threshold (plotted as dashed line)
        save: optional path to save figure
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for label, d in data_by_experiment.items():
        ax.plot(d['time'], d['S_k'], linewidth=1.2, label=label)
    
    ax.axhline(y=tau, color='red', linestyle='--', linewidth=1.5,
              label=f'Umbral τ = {tau}')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('$S_k$ (estadístico CUSUM)')
    ax.set_title('Evolución temporal del estadístico CUSUM')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    if save:
        fig.savefig(save, bbox_inches='tight')
    plt.show()

# Example:
# plot_cusum_evolution({
#     'E0 — Baseline': e0_data,
#     'E1 — Meaconing lento (0.1 m/s)': e1_data,
#     'E2 — Meaconing rápido (0.5 m/s)': e2_data,
# }, tau=2.0)

## 3. Time-To-Detect (TTD) vs Drift Velocity

Curva de sensibilidad: ¿cómo varía el tiempo de detección con la velocidad de arrastre?

In [ ]:
def plot_ttd_vs_drift(drift_velocities: list, ttd_values: list, save: str = None):
    """
    Plot TTD (Time-To-Detect) vs drift velocity.
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(drift_velocities, ttd_values, 'o-', markersize=8, linewidth=2)
    ax.set_xlabel('Drift velocity (m/s)')
    ax.set_ylabel('TTD — Time-To-Detect (s)')
    ax.set_title('Sensibilidad del detector: TTD vs velocidad de ataque')
    ax.grid(True, alpha=0.3)
    
    if save:
        fig.savefig(save, bbox_inches='tight')
    plt.show()

# Example:
# plot_ttd_vs_drift([0.05, 0.1, 0.2, 0.5, 1.0], [15.2, 8.1, 4.3, 2.1, 1.2])

## 4. Curva ROC

Trade-off TTD vs FAR para distintos valores del umbral τ.

In [ ]:
def plot_roc_curve(tau_values: list, ttd_values: list, far_values: list, save: str = None):
    """
    Plot ROC-like curve: TTD vs FAR for different τ values.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    scatter = ax.scatter(far_values, ttd_values, c=tau_values,
                        cmap='viridis', s=100, edgecolors='k')
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Umbral τ')
    
    # Annotate each point with its τ value
    for far, ttd, tau in zip(far_values, ttd_values, tau_values):
        ax.annotate(f'τ={tau:.1f}', (far, ttd),
                   textcoords='offset points', xytext=(5, 5), fontsize=9)
    
    ax.set_xlabel('FAR — False Alarm Rate (alarmas/min)')
    ax.set_ylabel('TTD — Time-To-Detect (s)')
    ax.set_title('Curva de trade-off TTD vs FAR')
    ax.grid(True, alpha=0.3)
    
    if save:
        fig.savefig(save, bbox_inches='tight')
    plt.show()

# Example:
# tau_vals = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
# ttds = [12.1, 8.3, 4.5, 3.2, 2.1, 1.5]
# fars = [2.3, 1.1, 0.2, 0.05, 0.01, 0.0]
# plot_roc_curve(tau_vals, ttds, fars)

## 5. Detector de umbral fijo vs CUSUM

Comparación de la innovación Δ(t) y el estadístico CUSUM S(t),
demostrando la ventaja del detector secuencial frente a un threshold fijo.

In [ ]:
def plot_threshold_vs_cusum(time: list, delta: list, S_k: list,
                           tau: float = 2.0, fixed_threshold: float = 2.0,
                           attack_start: float = None, save: str = None):
    """
    Compare fixed threshold detector vs CUSUM on the same data.
    Shows why CUSUM is superior for detecting persistent biases.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Top: innovation Δ(t) with fixed threshold
    ax1.plot(time, delta, linewidth=0.5, color='steelblue', alpha=0.7,
            label='Δ(t) = |D_GNSS − D_UWB|')
    ax1.axhline(y=fixed_threshold, color='orange', linestyle='--', linewidth=1.5,
               label=f'Umbral fijo = {fixed_threshold} m')
    ax1.set_ylabel('Δ(t) — Innovación (m)')
    ax1.set_title('Detector de umbral fijo — sensible al ruido')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Bottom: CUSUM S_k with τ line
    ax2.plot(time, S_k, linewidth=1.2, color='darkred',
            label='$S_k$ (estadístico CUSUM)')
    ax2.axhline(y=tau, color='red', linestyle='--', linewidth=1.5,
               label=f'Umbral τ = {tau}')
    ax2.fill_between(time, 0, tau, alpha=0.1, color='green',
                    label='Zona segura')
    ax2.fill_between(time, tau, max(S_k) * 1.1, alpha=0.1, color='red',
                    label='Zona de alarma')
    ax2.set_xlabel('Tiempo (s)')
    ax2.set_ylabel('$S_k$ — CUSUM')
    ax2.set_title('Detector CUSUM — acumula el sesgo persistente')
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    if attack_start is not None:
        for ax in [ax1, ax2]:
            ax.axvline(x=attack_start, color='purple', linestyle=':', linewidth=1.5,
                      label='Inicio del ataque')
    
    plt.tight_layout()
    if save:
        fig.savefig(save, bbox_inches='tight')
    plt.show()

# Example:
# plot_threshold_vs_cusum(time, delta, S_k, tau=2.0, fixed_threshold=2.0, attack_start=30.0)

## 6. Cálculo de métricas

Funciones auxiliares para calcular TTD y FAR a partir de los datos extraídos del rosbag.

In [ ]:
def compute_ttd(alert_times: list, attack_start_time: float) -> float:
    """
    Compute Time-To-Detect: time from attack activation to first alert.
    """
    post_attack = [t for t in alert_times if t >= attack_start_time]
    if not post_attack:
        return float('inf')  # Never detected
    return post_attack[0] - attack_start_time


def compute_far(alert_times: list, safe_duration: float) -> float:
    """
    Compute False Alarm Rate (alarms per minute) during safe period.
    """
    if safe_duration <= 0:
        return 0.0
    n_false_alarms = len(alert_times)
    return (n_false_alarms / safe_duration) * 60.0  # alarms/min

print('Métrica functions ready.')

---

## Experimentos

| Escenario | Duración | Ataque | Objetivo |
|---|---|---|---|
| E0 — Baseline | 10 min | Sin ataque | Medir FAR (debe ser ~0) |
| E1 — Meaconing lento | 10 min | drift 0.1 m/s | Medir TTD para ataque sutil |
| E2 — Meaconing rápido | 5 min | drift 0.5 m/s | Medir TTD para ataque obvio |
| E3 — Arranque en caliente | 10 min | Ataque ya activo desde t=0 | Medir TTD desde estado atacado |
| E4 — Variación de distancia entre robots | 10 min | drift 0.2 m/s | ¿Afecta la separación inicial al TTD? |

---

*Notebook generado para el TFM de Antonio García Alcón — Universidad Europea de Madrid, 2026.*